# 04 Modeling — Nikkei 225 Next-Day Return Prediction

## Purpose
Train and evaluate machine learning models that predict the next-day log return
of Nikkei 225 using the technical feature matrix from `03_technical.ipynb`.

## Motivation per model
| Model | Why |
|-------|-----|
| Ridge regression | Interpretable baseline; reveals linear feature weights; fast to tune |
| XGBoost | Captures non-linear interactions; handles mixed-scale features; provides feature importance |

## Evaluation strategy
- **Walk-forward cross-validation** (time-series split, no shuffling) to prevent data leakage.
- **Metrics**: RMSE, directional accuracy (sign of predicted vs actual return),
  Sharpe ratio of a simple long/short strategy based on the predicted sign.
- A model is considered useful if directional accuracy > 52% and strategy Sharpe > 0.

## Expected outputs
| Output | Location |
|--------|----------|
| Metric table (RMSE, dir. accuracy, Sharpe) per model | displayed |
| Cumulative return backtest chart | `output/figures/04_backtest.png` |
| XGBoost feature importance chart | `output/figures/04_feature_importance.png` |
| Trained models (Ridge + XGBoost) | `output/models/` |

## Sections
| # | Content |
|---|---------|
| 0 | Colab environment setup |
| 1 | Load feature matrix |
| 2 | Train/test split + scaling |
| 3 | Baseline: Ridge regression |
| 4 | XGBoost |
| 5 | Evaluation & comparison |
| 6 | Feature importance (XGBoost) |
| 7 | Simple backtest |
| 8 | Save models |


In [ ]:
# -- Step 1: Install packages -------------------------------------------
import subprocess, sys

pkgs = [
    'yfinance', 'pandas-ta',
    'scikit-learn>=1.5.0', 'statsmodels>=0.14.0',
    'xgboost>=2.0.0',
]
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + pkgs,
    capture_output=True, text=True
)
out = result.stdout + result.stderr
print(out.strip() if out.strip() else 'All packages already up to date.')

if 'Successfully installed' in out:
    print('\nPackages upgraded -- restarting runtime...')
    import os; os.kill(os.getpid(), 9)


In [ ]:
# -- Step 2: Repository & path setup ------------------------------------
import os, sys

REPO = '/content/Nikkei_Analysis'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists(REPO):
        !git clone https://github.com/Takumi-Itokawa-Finance/Nikkei_Analysis.git {REPO}
    else:
        !git -C {REPO} pull
    os.chdir(REPO)

    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = '/content/drive/MyDrive/Nikkei_Analysis/data'
    os.makedirs(DRIVE_DATA, exist_ok=True)
    if not os.path.exists(f'{REPO}/data'):
        os.symlink(DRIVE_DATA, f'{REPO}/data')

    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    print(f'Setup complete.  CWD={os.getcwd()}')
else:
    print('Running in local environment.')


## 1. Load Feature Matrix

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_FIGS   = 'output/figures'
OUTPUT_MODELS = 'output/models'
os.makedirs(OUTPUT_FIGS,   exist_ok=True)
os.makedirs(OUTPUT_MODELS, exist_ok=True)


In [ ]:
feat = pd.read_csv('data/processed/technical_features.csv',
                   index_col=0, parse_dates=True)

print(f'Shape  : {feat.shape}')
print(f'Period : {feat.index[0].date()} to {feat.index[-1].date()}')
display(feat.tail())


## 2. Train / Test Split + Scaling

Use the last 252 trading days (~1 year) as the hold-out test set.
All scaling is fit **only on the training set** to prevent leakage.


In [ ]:
from sklearn.preprocessing import StandardScaler

feature_cols = [c for c in feat.columns if c not in ['target_return', 'target_close']]
X = feat[feature_cols]
y = feat['target_return']

TEST_SIZE = 252
X_train, X_test = X.iloc[:-TEST_SIZE], X.iloc[-TEST_SIZE:]
y_train, y_test = y.iloc[:-TEST_SIZE], y.iloc[-TEST_SIZE:]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train.shape}  ({X_train.index[0].date()} to {X_train.index[-1].date()})')
print(f'Test : {X_test.shape}   ({X_test.index[0].date()}  to {X_test.index[-1].date()})')


## 3. Baseline: Ridge Regression

Ridge adds L2 regularisation to handle multicollinear features (many MAs are highly correlated).
Alpha is tuned via time-series cross-validation.


In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)
alphas = [0.01, 0.1, 1, 10, 100, 500]

ridge = RidgeCV(alphas=alphas, cv=tscv)
ridge.fit(X_train_s, y_train)

print(f'Best alpha: {ridge.alpha_}')

coef_df = pd.Series(ridge.coef_, index=feature_cols).sort_values(key=abs, ascending=False)
display(coef_df.head(15).to_frame('coefficient'))


## 4. XGBoost

Gradient-boosted trees capture non-linear relationships and feature interactions
that Ridge cannot (e.g. RSI extreme + volume surge).
Early stopping prevents overfitting.


In [ ]:
from xgboost import XGBRegressor

val_size = int(len(X_train) * 0.2)
X_tr, X_val = X_train_s[:-val_size], X_train_s[-val_size:]
y_tr, y_val = y_train.iloc[:-val_size], y_train.iloc[-val_size:]

xgb = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=30,
    eval_metric='rmse',
    random_state=42,
    verbosity=0,
)
xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
print(f'Best iteration: {xgb.best_iteration}')


## 5. Evaluation & Comparison

**Metrics**
- RMSE: raw prediction error in log-return units
- Directional accuracy: fraction of days where predicted sign matches actual sign
- Sharpe ratio: annualised Sharpe of a +1 / -1 position taken on predicted sign


In [ ]:
from sklearn.metrics import mean_squared_error

def evaluate(name, y_true, y_pred):
    rmse    = mean_squared_error(y_true, y_pred) ** 0.5
    dir_acc = (np.sign(y_pred) == np.sign(y_true)).mean()
    strategy = np.sign(y_pred) * y_true
    sharpe  = strategy.mean() / strategy.std() * np.sqrt(252)
    return {'Model': name, 'RMSE': rmse, 'Dir. Accuracy': dir_acc, 'Sharpe': sharpe}

ridge_pred = ridge.predict(X_test_s)
xgb_pred   = xgb.predict(X_test_s)
naive_pred = np.zeros(len(y_test))

results = pd.DataFrame([
    evaluate('Naive (zero)',  y_test, naive_pred),
    evaluate('Ridge',        y_test, ridge_pred),
    evaluate('XGBoost',      y_test, xgb_pred),
])
display(results.set_index('Model').style.format('{:.4f}'))


## 6. Feature Importance (XGBoost)

In [ ]:
importance = pd.Series(xgb.feature_importances_, index=feature_cols)\
               .sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
importance.head(20).plot(kind='barh', ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_title('XGBoost Feature Importance (top 20)')
ax.set_xlabel('Importance score')
plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/04_feature_importance.png', dpi=150)
plt.show()

display(importance.head(15).to_frame('importance'))


## 7. Simple Backtest

Simulate a long/short strategy: +1 when model predicts positive return, -1 otherwise.
No transaction costs or slippage -- directional signal test only.


In [ ]:
bt = pd.DataFrame({
    'actual':       y_test.values,
    'ridge_signal': np.sign(ridge_pred),
    'xgb_signal':   np.sign(xgb_pred),
}, index=y_test.index)

bt['ridge_pnl'] = bt['ridge_signal'] * bt['actual']
bt['xgb_pnl']   = bt['xgb_signal']   * bt['actual']
bt['buy_hold']   = bt['actual']

cum = bt[['ridge_pnl', 'xgb_pnl', 'buy_hold']].cumsum()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(cum.index, cum['buy_hold'],  label='Buy & Hold',  color='black',     linewidth=1.2)
ax.plot(cum.index, cum['ridge_pnl'], label='Ridge L/S',   color='steelblue', linewidth=1)
ax.plot(cum.index, cum['xgb_pnl'],  label='XGBoost L/S', color='orange',    linewidth=1)
ax.axhline(0, color='grey', linewidth=0.6, linestyle='--')
ax.set_title('Cumulative Log Return -- Backtest (test period)')
ax.set_ylabel('Cumulative log return')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/04_backtest.png', dpi=150)
plt.show()


## 8. Save Models

In [ ]:
import pickle

with open(f'{OUTPUT_MODELS}/ridge.pkl', 'wb') as f:
    pickle.dump({'model': ridge, 'scaler': scaler, 'features': feature_cols}, f)

xgb.save_model(f'{OUTPUT_MODELS}/xgboost.json')

print(f'Saved: {OUTPUT_MODELS}/ridge.pkl')
print(f'Saved: {OUTPUT_MODELS}/xgboost.json')
print('\nNext: review feature importance and consider adding macro features from 02_correlation.')
